## Preprocessor
Loads raw game data, applies quality filters, caps Stockfish evaluations, and labels blunders. Outputs `cleaned_games.parquet` for feature engineering.


In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))

from config import RAW_GAMES_PATH, CLEANED_PATH, MIN_ELO, MAX_ELO, TIME_CONTROLS, SNAPSHOT_MOVE, BLUNDER_WINDOW, BLUNDER_THRESHOLD, CAP

### Load raw data

In [ ]:
df = pd.read_parquet(RAW_GAMES_PATH)
df.shape

(15000, 8)

In [3]:
df.head()

,game_id,white_elo,black_elo,time_control,opening,result,moves,evals
0,https://lichess.org/oXyWZjvn,1802,1791,600+0,Sicilian Defense,1-0,"[e2e4, c7c5, g1f3, d7d6, b1c3, g8f6, f1c4, g7g...","[-32.0, 29.0, -27.0, 30.0, -29.0, 29.0, -19.0,..."
1,https://lichess.org/clKpvLl6,1270,1337,600+0,Scandinavian Defense,1-0,"[e2e4, d7d5, b1c3, d5d4, c3b5, c7c6, b5a3, g8f...","[-31.0, 72.0, 25.0, -18.0, 69.0, -18.0, 25.0, ..."
2,https://lichess.org/9mojvGgM,1743,1735,600+0,Pirc Defense,1-0,"[e2e4, d7d6, d2d4, g8f6, f1d3, g7g6, g1e2, f8g...","[-34.0, 55.0, -58.0, 39.0, -37.0, 64.0, -41.0,..."
3,https://lichess.org/cVGKAdOv,1822,1833,600+0,Philidor Defense: Exchange Variation,1/2-1/2,"[e2e4, e7e5, g1f3, d7d6, d2d4, e5d4, f3d4, g8f...","[-25.0, 31.0, -21.0, 20.0, -25.0, 33.0, -33.0,..."
4,https://lichess.org/3xkGeM1l,1195,1193,600+0,Elephant Gambit,1-0,"[e2e4, e7e5, g1f3, d7d5, e4d5, d8d5, b1c3, d5c...","[-16.0, 19.0, -34.0, 85.0, -89.0, 93.0, -80.0,..."


### Filter and clean
Strip increment from time control and keep only 600/900/1800s base times.


In [4]:
# Splitting by time control and filtering

df['time_control'] = df['time_control'].str.split('+').str[0].astype(int)
df = df[df['time_control'].isin(TIME_CONTROLS)]
print(f"After filtering by time control, dataset shape: {df.shape}")

After filtering by time control, dataset shape: (15000, 8)


Remove games with fewer than 25 moves — too short to have a meaningful blunder window.


In [5]:
# Filter short games

df = df[df['moves'].apply(len) >= 25]
print(f"After filtering by minimum number of moves, dataset shape: {df.shape}")

After filtering by minimum number of moves, dataset shape: (14556, 8)


Remove games with missing Stockfish evals in the critical window (moves 15–25), as these can't be reliably labeled.


In [6]:
# Filter games with missing evals in critical window

def has_missing_evals(evals, start=SNAPSHOT_MOVE, end=SNAPSHOT_MOVE + BLUNDER_WINDOW):
    window = evals[start:end]
    return any(v != v for v in window)  # Check for NaN

df = df[~df['evals'].apply(has_missing_evals)]
print(f"After filtering games with missing evals in critical window, dataset shape: {df.shape}")

After filtering games with missing evals in critical window, dataset shape: (14541, 8)


### Cap evaluations
Clip all eval values to ±1000 centipawns to neutralize mate score sentinels (±10000) which would cause artificial swings when computing deltas.


In [7]:
# cap evals

def cap_evals(evals):
    return np.array([max(-CAP, min(CAP, v)) if v == v else np.nan for v in evals])  # Preserve NaN

df['evals'] = df['evals'].apply(cap_evals)
print("Evals capped at ±1000.")

Evals capped at ±1000.


### Label blunders
A game is labeled a blunder if either side drops >200cp in a single move within the window (moves 15–25), evaluated per-side to account for alternating eval perspective.


In [8]:
# label blunders

def has_blunder(evals, start=SNAPSHOT_MOVE, end=SNAPSHOT_MOVE + BLUNDER_WINDOW, threshold=BLUNDER_THRESHOLD):
    window = [v for v in evals[start:end] if v == v] # filter out nans
    if len(window) < 2:
        return 0
    for offset in [0,1]:
        side = window[offset::2]
        for i in range(1, len(side)):
            if side[i-1] - side[i] > threshold:
                return 1
    return 0

df['blunder_label'] = df['evals'].apply(has_blunder)
print(f"Blunder rate: {df['blunder_label'].mean():.2%}")
print(f"Class balance = blunder: {df['blunder_label'].sum()}, no blunder: {(df['blunder_label']==0).sum()}")

Blunder rate: 25.99%
Class balance = blunder: 3779, no blunder: 10762


### Save

In [ ]:
CLEANED_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(CLEANED_PATH, index=False)
print(f"Saved to {CLEANED_PATH}")
print(f"Final shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Saved to D:\DS_ML_journey\chess_blunder\data\processed\cleaned_games.parquet
Final shape: (14541, 9)
Columns: ['game_id', 'white_elo', 'black_elo', 'time_control', 'opening', 'result', 'moves', 'evals', 'blunder_label']
